In [1]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # from budget.py
from humaidclf.batch import use_api_key_env           # context manager for key switching
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["train"]             # or ["train","dev","test"]
MODEL = "gpt-4o-mini"
RULES = RULES_1
TAG = "modeS-RULES1"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

BATCH_TOKEN_LIMIT = 2_000_000  # Tier-1 cap
SAFETY_MARGIN = 0.90           # 10% headroom
MAX_OUTPUT_TOKENS = 40

In [2]:
# --- discover datasets ---
def discover_tsvs(base: Path, splits: list[str]):
    items = []
    for event_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        event = event_dir.name
        for split in splits:
            tsv = event_dir / f"{event}_{split}.tsv"
            if tsv.exists():
                items.append({"event": event, "split": split, "tsv": str(tsv)})
    return pd.DataFrame(items)

df_sources = discover_tsvs(BASE, SPLITS)

df_sources

,event,split,tsv
0,california_wildfires_2018,train,Dataset\HumAID\california_wildfires_2018\calif...
1,canada_wildfires_2016,train,Dataset\HumAID\canada_wildfires_2016\canada_wi...
2,cyclone_idai_2019,train,Dataset\HumAID\cyclone_idai_2019\cyclone_idai_...
3,hurricane_dorian_2019,train,Dataset\HumAID\hurricane_dorian_2019\hurricane...
4,hurricane_florence_2018,train,Dataset\HumAID\hurricane_florence_2018\hurrica...
5,hurricane_harvey_2017,train,Dataset\HumAID\hurricane_harvey_2017\hurricane...
6,hurricane_irma_2017,train,Dataset\HumAID\hurricane_irma_2017\hurricane_i...
7,hurricane_maria_2017,train,Dataset\HumAID\hurricane_maria_2017\hurricane_...
8,kaikoura_earthquake_2016,train,Dataset\HumAID\kaikoura_earthquake_2016\kaikou...
9,kerala_floods_2018,train,Dataset\HumAID\kerala_floods_2018\kerala_flood...


In [3]:
from pathlib import Path
import pandas as pd
from typing import List
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index
from humaidclf.batch import use_api_key_env
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS: List[str] = ["train"]  # or ["train","dev","test"]

# --- discover datasets ---
def discover_tsvs(base: Path, splits: List[str]) -> pd.DataFrame:
    if not base.exists():
        print(f"[WARN] Base path not found: {base.resolve()}")
        return pd.DataFrame(columns=["event", "split", "tsv"])
    items = []
    for event_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        event = event_dir.name
        for split in splits:
            tsv = event_dir / f"{event}_{split}.tsv"
            if tsv.exists():
                items.append({"event": event, "split": split, "tsv": str(tsv)})
            else:
                print(f"[WARN] Missing file: {tsv}")
    return pd.DataFrame(items)

# --- helpers to run a list of datasets ---
def show_class_distribution(dflist: pd.DataFrame, label_col: str = "class_label"):
    if dflist.empty:
        print("[INFO] No datasets discovered.")
        return
    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        print(f"\n=== Showing {event}/{split} ===")
        try:
            df_current_even = pd.read_csv(tsv, sep="\t")

            # Clean labels (drop NaN, trim whitespace) for class counting
            labels_clean = (
                df_current_even[label_col]
                .dropna()
                .astype(str)
                .str.strip()
            )

            # Print distribution (using cleaned labels)
            counts = labels_clean.value_counts()
            print(counts.to_string())

            # Print total number of classes
            n_classes = labels_clean.nunique()
            print(f"Total classes: {n_classes}")

        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")

df_sources = discover_tsvs(BASE, SPLITS)
show_class_distribution(df_sources)


=== Showing california_wildfires_2018/train ===
class_label
injured_or_dead_people                    1362
rescue_volunteering_or_donation_effort     991
not_humanitarian                           923
other_relevant_information                 727
sympathy_and_support                       330
infrastructure_and_utility_damage          295
displaced_people_and_evacuations           258
missing_or_found_people                    125
caution_and_advice                          97
requests_or_urgent_needs                    55
Total classes: 10

=== Showing canada_wildfires_2016/train ===
class_label
rescue_volunteering_or_donation_effort    653
displaced_people_and_evacuations          266
other_relevant_information                218
infrastructure_and_utility_damage         176
sympathy_and_support                      113
caution_and_advice                         74
not_humanitarian                           55
requests_or_urgent_needs                   14
Total classes: 8

=== Show

In [4]:
# --- config ---
BASE_TEST = Path("Dataset/HumAID")
SPLITS_TEST: List[str] = ["test"]  # or ["train","dev","test"]

df_sources_test = discover_tsvs(BASE_TEST, SPLITS_TEST)
show_class_distribution(df_sources_test)


=== Showing california_wildfires_2018/test ===
class_label
injured_or_dead_people                    385
rescue_volunteering_or_donation_effort    280
not_humanitarian                          261
other_relevant_information                205
sympathy_and_support                       94
infrastructure_and_utility_damage          84
displaced_people_and_evacuations           72
missing_or_found_people                    36
caution_and_advice                         28
requests_or_urgent_needs                   16
Total classes: 10

=== Showing canada_wildfires_2016/test ===
class_label
rescue_volunteering_or_donation_effort    186
displaced_people_and_evacuations           75
other_relevant_information                 61
infrastructure_and_utility_damage          50
sympathy_and_support                       32
caution_and_advice                         21
not_humanitarian                           16
requests_or_urgent_needs                    4
Total classes: 8

=== Showing cyclone_